<a href="https://colab.research.google.com/github/fethinho/AI-Trader/blob/main/FETH%C4%B0NHO_AI_OKX_tarama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install ccxt pandas numpy requests schedule

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 14.2 MB/s eta 0:00:00


In [6]:
import ccxt
import pandas as pd
import numpy as np
import requests
import time
from datetime import datetime

# ============================================================
# ⚙️ AYARLAR - Buraya kendi bilgilerini gir
# ============================================================
TELEGRAM_BOT_TOKEN = "8460458441:AAG81QuP0nDc0AT5SYON63bus2d0udab0Iw"   # @BotFather'dan al
TELEGRAM_CHAT_ID   = "314746106"     # @userinfobot'tan al

EXCHANGE_ID   = "bybit"
QUOTE_ASSET   = "USDT"
TOP_N_SYMBOLS = 100

RSI_MIN    = 40
RSI_MAX    = 89
RSI_PERIOD = 14
EMA_FAST   = 20
EMA_SLOW   = 50

TIMEFRAMES = {
    "1h": "Saatlik",
    "1d": "Günlük",
}
# ============================================================

def send_telegram(message: str):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {"chat_id": TELEGRAM_CHAT_ID, "text": message, "parse_mode": "HTML"}
    try:
        r = requests.post(url, json=payload, timeout=10)
        r.raise_for_status()
    except Exception as e:
        print(f"[TG HATA] {e}")

def calc_rsi(closes: pd.Series, period: int = 14) -> float:
    delta    = closes.diff()
    gain     = delta.where(delta > 0, 0.0)
    loss     = -delta.where(delta < 0, 0.0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs  = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return round(float(rsi.iloc[-1]), 2)

def calc_ema(closes: pd.Series, period: int) -> pd.Series:
    return closes.ewm(span=period, adjust=False).mean()

def ema_crossover(ema_fast: pd.Series, ema_slow: pd.Series) -> bool:
    return (ema_fast.iloc[-2] <= ema_slow.iloc[-2]) and \
           (ema_fast.iloc[-1]  > ema_slow.iloc[-1])

def fetch_ohlcv(exchange, symbol: str, timeframe: str, limit: int = 200):
    try:
        raw = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
        if len(raw) < EMA_SLOW + 5:
            return None
        df = pd.DataFrame(raw, columns=["ts","open","high","low","close","volume"])
        df["close"]  = df["close"].astype(float)
        df["volume"] = df["volume"].astype(float)
        return df
    except Exception:
        return None

def scan(exchange, symbols: list, timeframe: str) -> list:
    results = []
    for symbol in symbols:
        df = fetch_ohlcv(exchange, symbol, timeframe)
        if df is None:
            continue

        closes   = df["close"]
        volumes  = df["volume"]
        ema20    = calc_ema(closes, EMA_FAST)
        ema50    = calc_ema(closes, EMA_SLOW)
        rsi      = calc_rsi(closes, RSI_PERIOD)
        price    = float(closes.iloc[-1])
        vol_last = float(volumes.iloc[-1])
        vol_avg  = float(volumes.iloc[-20:].mean())

        cond_price  = price > float(ema20.iloc[-1])
        cond_rsi    = RSI_MIN <= rsi <= RSI_MAX
        cond_cross  = ema_crossover(ema20, ema50)
        cond_volume = vol_last > vol_avg

        if cond_price and cond_rsi and cond_cross and cond_volume:
            results.append({
                "symbol": symbol,
                "price":  price,
                "ema20":  round(float(ema20.iloc[-1]), 6),
                "ema50":  round(float(ema50.iloc[-1]), 6),
                "rsi":    rsi,
                "vol_x":  round(vol_last / vol_avg, 2),
            })
        time.sleep(0.05)
    return results

def format_message(results: list, timeframe_label: str, tf: str) -> str:
    now = datetime.now().strftime("%d.%m.%Y %H:%M")
    if not results:
        return (
            f"🔍 <b>{timeframe_label} Tarama</b> [{tf}]\n"
            f"📅 {now}\n\n"
            f"❌ Kriterlere uyan coin bulunamadı."
        )
    lines = [
        f"🚀 <b>{timeframe_label} Tarama</b> [{tf}]",
        f"📅 {now}",
        f"✅ <b>{len(results)} sinyal</b>\n",
        f"{'Sembol':<14} {'Fiyat':>12} {'RSI':>6} {'Hacim X':>8}",
        "─" * 44,
    ]
    for r in results:
        lines.append(
            f"{r['symbol']:<14} {r['price']:>12.4f} {r['rsi']:>6.1f} {r['vol_x']:>7.1f}x"
        )
    lines.append("\n📌 <i>Fiyat&gt;EMA20 | RSI 40-89 | EMA20xEMA50 | Hacim&gt;Ort</i>")
    return "\n".join(lines)

def main():
    print(f"🤖 Kripto Tarayıcı Başladı — {datetime.now().strftime('%d.%m.%Y %H:%M')}")
    exchange = getattr(ccxt, EXCHANGE_ID)({"enableRateLimit": True})
    exchange.load_markets()

    symbols = [
    s for s, m in exchange.markets.items()
    if m.get("quote") == QUOTE_ASSET
    and m.get("active")
    and "/" in s
    and ":" not in s   # futures/perp hariç
    and m.get("type", "spot") == "spot"
]

    try:
        tickers = exchange.fetch_tickers(symbols[:300])
        sorted_syms = sorted(
            [s for s in tickers if tickers[s].get("quoteVolume")],
            key=lambda s: tickers[s]["quoteVolume"],
            reverse=True,
        )[:TOP_N_SYMBOLS]
    except Exception:
        sorted_syms = symbols[:TOP_N_SYMBOLS]

    print(f"  {len(sorted_syms)} coin taranıyor...")

    for tf, label in TIMEFRAMES.items():
        print(f"\n  [{label}] taranıyor...")
        hits = scan(exchange, sorted_syms, tf)
        msg  = format_message(hits, label, tf)
        print(msg)
        send_telegram(msg)
        print(f"  ✅ Telegram'a gönderildi.")
        time.sleep(1)

    print("\n✅ Tarama tamamlandı.")

main()

🤖 Kripto Tarayıcı Başladı — 23.04.2026 08:11


RateLimitExceeded: bybit GET https://api.bybit.com/v5/market/instruments-info?category=spot 403 Forbidden {
    error:The Amazon CloudFront distribution is configured to block access from your country
}

In [7]:
# GitHub'daki kendi ccxt fork'unu kullan (veya resmi)
!pip install ccxt pandas requests -q

In [8]:
import ccxt
import pandas as pd
import requests
import time
from datetime import datetime

# ============================================================
# ⚙️ AYARLAR
# ============================================================
TELEGRAM_BOT_TOKEN = "8460458441:AAG81QuP0nDc0AT5SYON63bus2d0udab0Iw"      # @BotFather
TELEGRAM_CHAT_ID   = "314746106"   # @userinfobot

EXCHANGE_ID   = "okx"       # ✅ Colab'da çalışır
QUOTE_ASSET   = "USDT"
TOP_N_SYMBOLS = 80

RSI_MIN    = 40
RSI_MAX    = 89
RSI_PERIOD = 14
EMA_FAST   = 20
EMA_SLOW   = 50

TIMEFRAMES = {"1H": "Saatlik", "1D": "Günlük"}
# ============================================================

def send_telegram(msg):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        requests.post(url, json={"chat_id": TELEGRAM_CHAT_ID, "text": msg, "parse_mode": "HTML"}, timeout=10)
    except Exception as e:
        print(f"TG Hata: {e}")

def calc_rsi(closes, period=14):
    delta = closes.diff()
    gain  = delta.where(delta > 0, 0.0).ewm(com=period-1, min_periods=period).mean()
    loss  = (-delta.where(delta < 0, 0.0)).ewm(com=period-1, min_periods=period).mean()
    return round(float((100 - 100/(1 + gain/loss)).iloc[-1]), 2)

def calc_ema(closes, period):
    return closes.ewm(span=period, adjust=False).mean()

def ema_cross(fast, slow):
    return (fast.iloc[-2] <= slow.iloc[-2]) and (fast.iloc[-1] > slow.iloc[-1])

def get_ohlcv(exchange, symbol, tf, limit=200):
    try:
        raw = exchange.fetch_ohlcv(symbol, timeframe=tf, limit=limit)
        if len(raw) < 60:
            return None
        df = pd.DataFrame(raw, columns=["ts","open","high","low","close","volume"])
        df[["close","volume"]] = df[["close","volume"]].astype(float)
        return df
    except:
        return None

def scan_all(exchange, symbols, tf, label):
    hits = []
    print(f"  {label} taranıyor ({len(symbols)} coin)...")
    for sym in symbols:
        df = get_ohlcv(exchange, sym, tf)
        if df is None:
            continue
        c = df["close"]
        v = df["volume"]
        ema20 = calc_ema(c, EMA_FAST)
        ema50 = calc_ema(c, EMA_SLOW)
        try:
            rsi = calc_rsi(c)
        except:
            continue
        price    = float(c.iloc[-1])
        vol_last = float(v.iloc[-1])
        vol_avg  = float(v.iloc[-20:].mean())

        if (price > float(ema20.iloc[-1]) and
            RSI_MIN <= rsi <= RSI_MAX and
            ema_cross(ema20, ema50) and
            vol_last > vol_avg):
            hits.append({
                "symbol": sym,
                "price":  price,
                "rsi":    rsi,
                "vol_x":  round(vol_last/vol_avg, 2)
            })
        time.sleep(0.08)

    now = datetime.now().strftime("%d.%m.%Y %H:%M")
    if not hits:
        msg = f"🔍 <b>{label} Tarama</b> [{tf}]\n📅 {now}\n\n❌ Sinyal yok."
    else:
        lines = [f"🚀 <b>{label} Tarama</b> [{tf}]", f"📅 {now}", f"✅ <b>{len(hits)} sinyal</b>\n"]
        for h in hits:
            lines.append(f"<b>{h['symbol']}</b>  Fiyat:{h['price']:.4f}  RSI:{h['rsi']}  Hacim:{h['vol_x']}x")
        lines.append("\n📌 <i>Fiyat&gt;EMA20 | RSI 40-89 | EMA20xEMA50 | Hacim&gt;Ort</i>")
        msg = "\n".join(lines)

    print(msg)
    send_telegram(msg)
    return hits

# ─── BAŞLAT ───────────────────────────────────────────────
print(f"🤖 Kripto Tarayıcı — {datetime.now().strftime('%d.%m.%Y %H:%M')}")
print(f"   Borsa: {EXCHANGE_ID.upper()} | {TOP_N_SYMBOLS} coin")

exchange = getattr(ccxt, EXCHANGE_ID)({"enableRateLimit": True})
exchange.load_markets()

# Spot USDT pariteleri — hacme göre sırala
all_syms = [
    s for s, m in exchange.markets.items()
    if m.get("quote") == QUOTE_ASSET
    and m.get("active")
    and m.get("type","spot") == "spot"
    and "/" in s and ":" not in s
]

try:
    tickers = exchange.fetch_tickers(all_syms[:300])
    syms = sorted(
        [s for s in tickers if tickers[s].get("quoteVolume")],
        key=lambda s: tickers[s]["quoteVolume"], reverse=True
    )[:TOP_N_SYMBOLS]
except:
    syms = all_syms[:TOP_N_SYMBOLS]

print(f"   {len(syms)} coin seçildi.\n")

for tf, label in TIMEFRAMES.items():
    scan_all(exchange, syms, tf, label)
    time.sleep(2)

print("\n✅ Tarama tamamlandı!")

🤖 Kripto Tarayıcı — 23.04.2026 08:16
   Borsa: OKX | 80 coin
   80 coin seçildi.

  Saatlik taranıyor (80 coin)...
🔍 <b>Saatlik Tarama</b> [1H]
📅 23.04.2026 08:16

❌ Sinyal yok.
  Günlük taranıyor (80 coin)...
🔍 <b>Günlük Tarama</b> [1D]
📅 23.04.2026 08:16

❌ Sinyal yok.

✅ Tarama tamamlandı!


In [10]:
import ccxt
import pandas as pd
import requests
import time
from datetime import datetime

# ============================================================
TELEGRAM_BOT_TOKEN = "8460458441:AAG81QuP0nDc0AT5SYON63bus2d0udab0Iw"
TELEGRAM_CHAT_ID   = "314746106"

EXCHANGE_ID   = "okx"
QUOTE_ASSET   = "USDT"
TOP_N_SYMBOLS = 200   # ← daha fazla coin

RSI_MIN    = 40
RSI_MAX    = 89
RSI_PERIOD = 14
EMA_FAST   = 20
EMA_SLOW   = 50

TIMEFRAMES = {"1h": "Saatlik", "1d": "Günlük"}
# ============================================================

def send_telegram(msg):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        r = requests.post(url, json={
            "chat_id": TELEGRAM_CHAT_ID,
            "text": msg,
            "parse_mode": "Markdown"   # ← HTML yerine Markdown
        }, timeout=10)
        print("  📨 TG:", r.status_code)
    except Exception as e:
        print(f"  TG Hata: {e}")

def calc_rsi(closes, period=14):
    delta = closes.diff()
    gain  = delta.where(delta > 0, 0.0).ewm(com=period-1, min_periods=period).mean()
    loss  = (-delta.where(delta < 0, 0.0)).ewm(com=period-1, min_periods=period).mean()
    rs = gain / loss.replace(0, 1e-10)
    return round(float((100 - 100/(1+rs)).iloc[-1]), 2)

def calc_ema(closes, period):
    return closes.ewm(span=period, adjust=False).mean()

def get_ohlcv(exchange, symbol, tf, limit=200):
    try:
        raw = exchange.fetch_ohlcv(symbol, timeframe=tf, limit=limit)
        if len(raw) < 60:
            return None
        df = pd.DataFrame(raw, columns=["ts","open","high","low","close","volume"])
        df[["close","volume"]] = df[["close","volume"]].astype(float)
        return df
    except:
        return None

def scan_all(exchange, symbols, tf, label):
    hits = []
    skipped = 0
    print(f"\n  [{label}] taranıyor — {len(symbols)} coin...")

    for sym in symbols:
        df = get_ohlcv(exchange, sym, tf)
        if df is None:
            skipped += 1
            continue

        c = df["close"]
        v = df["volume"]
        ema20 = calc_ema(c, EMA_FAST)
        ema50 = calc_ema(c, EMA_SLOW)

        try:
            rsi = calc_rsi(c)
        except:
            continue

        price    = float(c.iloc[-1])
        vol_last = float(v.iloc[-1])
        vol_avg  = float(v.iloc[-20:].mean())

        # ✅ Kriterler (crossover yerine pozisyon — daha fazla sinyal)
        cond1 = price > float(ema20.iloc[-1])             # Fiyat > EMA20
        cond2 = RSI_MIN <= rsi <= RSI_MAX                  # RSI 40-89
        cond3 = float(ema20.iloc[-1]) > float(ema50.iloc[-1])  # EMA20 > EMA50 (trend yukarı)
        cond4 = vol_last > vol_avg                         # Hacim > Ortalama

        if cond1 and cond2 and cond3 and cond4:
            hits.append({
                "symbol": sym.replace("/USDT",""),
                "price":  price,
                "rsi":    rsi,
                "vol_x":  round(vol_last / vol_avg, 2)
            })

        time.sleep(0.06)

    print(f"  → {len(hits)} sinyal bulundu, {skipped} coin atlandı")

    now = datetime.now().strftime("%d.%m.%Y %H:%M")

    if not hits:
        msg = f"🔍 *{label} Tarama* [{tf}]\n📅 {now}\n\n❌ Sinyal yok."
    else:
        lines = [
            f"🚀 *{label} Tarama* [{tf}]",
            f"📅 {now}",
            f"✅ *{len(hits)} sinyal*\n",
            f"{'Coin':<12} {'Fiyat':>10} {'RSI':>5} {'Hcm':>5}",
            "─────────────────────────────",
        ]
        for h in sorted(hits, key=lambda x: x["rsi"]):
            lines.append(f"{h['symbol']:<12} {h['price']:>10.4f} {h['rsi']:>5.1f} {h['vol_x']:>4.1f}x")
        lines.append(f"\n📌 Fiyat>EMA20 | RSI {RSI_MIN}-{RSI_MAX} | EMA20>EMA50 | Hacim>Ort")
        msg = "\n".join(lines)

    print(msg)
    send_telegram(msg)
    return hits

# ─── BAŞLAT ───────────────────────────────────────────────
print(f"🤖 Kripto Tarayıcı — {datetime.now().strftime('%d.%m.%Y %H:%M')}")

exchange = getattr(ccxt, EXCHANGE_ID)({"enableRateLimit": True})
exchange.load_markets()

all_syms = [
    s for s, m in exchange.markets.items()
    if m.get("quote") == QUOTE_ASSET
    and m.get("active")
    and m.get("type","spot") == "spot"
    and "/" in s and ":" not in s
]
print(f"   Toplam spot çift: {len(all_syms)}")

# Hacme göre sırala, en yüksek TOP_N al
try:
    tickers = exchange.fetch_tickers(all_syms[:400])
    syms = sorted(
        [s for s in tickers if tickers[s].get("quoteVolume")],
        key=lambda s: tickers[s]["quoteVolume"], reverse=True
    )[:TOP_N_SYMBOLS]
except:
    syms = all_syms[:TOP_N_SYMBOLS]

print(f"   {len(syms)} coin taranacak (en yüksek hacimli)\n")

for tf, label in TIMEFRAMES.items():
    scan_all(exchange, syms, tf, label)
    time.sleep(3)

print("\n✅ Tarama tamamlandı!")

🤖 Kripto Tarayıcı — 23.04.2026 08:24
   Toplam spot çift: 292
   200 coin taranacak (en yüksek hacimli)


  [Saatlik] taranıyor — 200 coin...
  → 5 sinyal bulundu, 0 coin atlandı
🚀 *Saatlik Tarama* [1h]
📅 23.04.2026 08:25
✅ *5 sinyal*

Coin              Fiyat   RSI   Hcm
─────────────────────────────
ZRX              0.1122  55.2  1.5x
GMX              7.0560  66.6  1.8x
OL               0.0083  66.9  2.0x
MASK             0.5013  69.7  4.0x
NMR              9.4960  70.7  2.3x

📌 Fiyat>EMA20 | RSI 40-89 | EMA20>EMA50 | Hacim>Ort
  📨 TG: 200

  [Günlük] taranıyor — 200 coin...
  → 3 sinyal bulundu, 5 coin atlandı
🚀 *Günlük Tarama* [1d]
📅 23.04.2026 08:26
✅ *3 sinyal*

Coin              Fiyat   RSI   Hcm
─────────────────────────────
ZBCN             0.0029  57.9  1.1x
ZRX              0.1123  58.0  1.1x
BIO              0.0323  62.7  1.2x

📌 Fiyat>EMA20 | RSI 40-89 | EMA20>EMA50 | Hacim>Ort
  📨 TG: 200

✅ Tarama tamamlandı!


In [11]:
import ccxt
import pandas as pd
import requests
import time
from datetime import datetime, timezone, timedelta

# ============================================================
# ⚙️ AYARLAR
# ============================================================
TELEGRAM_BOT_TOKEN = "8460458441:AAG81QuP0nDc0AT5SYON63bus2d0udab0Iw"
TELEGRAM_CHAT_ID   = "314746106"

EXCHANGE_ID   = "okx"
QUOTE_ASSET   = "USDT"
TOP_N_SYMBOLS = 200

RSI_MIN    = 40
RSI_MAX    = 89
RSI_PERIOD = 14
EMA_FAST   = 20
EMA_SLOW   = 50

TIMEFRAMES = {"1h": "Saatlik", "1d": "Günlük"}   # ✅ küçük harf

TZ_TR = timezone(timedelta(hours=3))   # UTC+3 İstanbul
# ============================================================

def now_tr():
    return datetime.now(TZ_TR).strftime("%d.%m.%Y %H:%M")

def send_telegram(msg):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        r = requests.post(url, json={
            "chat_id":    TELEGRAM_CHAT_ID,
            "text":       msg,
            "parse_mode": "Markdown"
        }, timeout=10)
        print(f"  📨 TG: {r.status_code}")
    except Exception as e:
        print(f"  TG Hata: {e}")

def calc_rsi(closes, period=14):
    delta = closes.diff()
    gain  = delta.where(delta > 0, 0.0).ewm(com=period-1, min_periods=period).mean()
    loss  = (-delta.where(delta < 0, 0.0)).ewm(com=period-1, min_periods=period).mean()
    rs    = gain / loss.replace(0, 1e-10)
    return round(float((100 - 100 / (1 + rs)).iloc[-1]), 2)

def calc_ema(closes, period):
    return closes.ewm(span=period, adjust=False).mean()

def get_ohlcv(exchange, symbol, tf, limit=200):
    try:
        raw = exchange.fetch_ohlcv(symbol, timeframe=tf, limit=limit)
        if len(raw) < 60:
            return None
        df = pd.DataFrame(raw, columns=["ts","open","high","low","close","volume"])
        df[["close","volume"]] = df[["close","volume"]].astype(float)
        return df
    except Exception as e:
        return None

def scan_all(exchange, symbols, tf, label):
    hits = []
    print(f"\n  [{label}] taranıyor — {len(symbols)} coin...")

    for sym in symbols:
        df = get_ohlcv(exchange, sym, tf)
        if df is None:
            continue

        c = df["close"]
        v = df["volume"]
        ema20 = calc_ema(c, EMA_FAST)
        ema50 = calc_ema(c, EMA_SLOW)

        try:
            rsi = calc_rsi(c)
        except:
            continue

        price    = float(c.iloc[-1])
        vol_last = float(v.iloc[-1])
        vol_avg  = float(v.iloc[-20:].mean())

        cond1 = price > float(ema20.iloc[-1])
        cond2 = RSI_MIN <= rsi <= RSI_MAX
        cond3 = float(ema20.iloc[-1]) > float(ema50.iloc[-1])
        cond4 = vol_last > vol_avg

        if cond1 and cond2 and cond3 and cond4:
            hits.append({
                "symbol": sym.replace("/USDT", ""),
                "price":  price,
                "rsi":    rsi,
                "vol_x":  round(vol_last / vol_avg, 2)
            })
        time.sleep(0.06)

    print(f"  → {len(hits)} sinyal bulundu")

    zaman = now_tr()

    if not hits:
        msg = (
            f"━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"🤖 *FETHİNHO AI* | OKX Tarama\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"📊 *{label} Tarama* `[{tf}]`\n"
            f"🕐 {zaman} \\(UTC\\+3\\)\n\n"
            f"❌ Kriterlere uyan coin bulunamadı\\."
        )
    else:
        lines = [
            "━━━━━━━━━━━━━━━━━━━━━━━",
            "🤖 *FETHİNHO AI* | OKX Tarama",
            "━━━━━━━━━━━━━━━━━━━━━━━",
            f"📊 *{label} Tarama* `[{tf}]`",
            f"🕐 {zaman} (UTC+3)",
            f"✅ *{len(hits)} Sinyal Bulundu*",
            "───────────────────────",
        ]
        for h in sorted(hits, key=lambda x: x["rsi"]):
            lines.append(
                f"🔸 *{h['symbol']}*  |  "
                f"💰 {h['price']:.4f}  |  "
                f"📈 RSI: {h['rsi']}  |  "
                f"🔥 {h['vol_x']}x"
            )
        lines += [
            "───────────────────────",
            f"📌 Fiyat>EMA{EMA_FAST} | RSI {RSI_MIN}\\-{RSI_MAX} | EMA{EMA_FAST}>EMA{EMA_SLOW} | Hacim>Ort",
            "━━━━━━━━━━━━━━━━━━━━━━━",
        ]
        msg = "\n".join(lines)

    print(msg)
    send_telegram(msg)
    return hits

# ─── BAŞLAT ───────────────────────────────────────────────
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"🤖 FETHİNHO AI — Kripto Tarayıcı")
print(f"🕐 {now_tr()} (UTC+3)")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━")

exchange = getattr(ccxt, EXCHANGE_ID)({"enableRateLimit": True})
exchange.load_markets()

all_syms = [
    s for s, m in exchange.markets.items()
    if m.get("quote") == QUOTE_ASSET
    and m.get("active")
    and m.get("type", "spot") == "spot"
    and "/" in s and ":" not in s
]
print(f"   Toplam spot çift: {len(all_syms)}")

try:
    tickers = exchange.fetch_tickers(all_syms[:400])
    syms = sorted(
        [s for s in tickers if tickers[s].get("quoteVolume")],
        key=lambda s: tickers[s]["quoteVolume"],
        reverse=True
    )[:TOP_N_SYMBOLS]
except:
    syms = all_syms[:TOP_N_SYMBOLS]

print(f"   {len(syms)} coin taranacak\n")

for tf, label in TIMEFRAMES.items():
    scan_all(exchange, syms, tf, label)
    time.sleep(3)

print(f"\n✅ Tarama tamamlandı! — {now_tr()}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 FETHİNHO AI — Kripto Tarayıcı
🕐 23.04.2026 11:29 (UTC+3)
━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Toplam spot çift: 292
   200 coin taranacak


  [Saatlik] taranıyor — 200 coin...
  → 5 sinyal bulundu
━━━━━━━━━━━━━━━━━━━━━━━
🤖 *FETHİNHO AI* | OKX Tarama
━━━━━━━━━━━━━━━━━━━━━━━
📊 *Saatlik Tarama* `[1h]`
🕐 23.04.2026 11:30 (UTC+3)
✅ *5 Sinyal Bulundu*
───────────────────────
🔸 *ZRX*  |  💰 0.1121  |  📈 RSI: 54.66  |  🔥 1.92x
🔸 *OL*  |  💰 0.0082  |  📈 RSI: 64.16  |  🔥 2.19x
🔸 *GMX*  |  💰 7.0560  |  📈 RSI: 66.59  |  🔥 2.1x
🔸 *MASK*  |  💰 0.5026  |  📈 RSI: 70.58  |  🔥 4.25x
🔸 *NMR*  |  💰 9.5380  |  📈 RSI: 71.69  |  🔥 2.32x
───────────────────────
📌 Fiyat>EMA20 | RSI 40\-89 | EMA20>EMA50 | Hacim>Ort
━━━━━━━━━━━━━━━━━━━━━━━
  📨 TG: 200

  [Günlük] taranıyor — 200 coin...
  → 3 sinyal bulundu
━━━━━━━━━━━━━━━━━━━━━━━
🤖 *FETHİNHO AI* | OKX Tarama
━━━━━━━━━━━━━━━━━━━━━━━
📊 *Günlük Tarama* `[1d]`
🕐 23.04.2026 11:31 (UTC+3)
✅ *3 Sinyal Bulundu*
───────────────────────
🔸 *ZRX*  | 

In [12]:
import ccxt
import pandas as pd
import requests
import time
from datetime import datetime, timezone, timedelta

# ============================================================
TELEGRAM_BOT_TOKEN = "8460458441:AAG81QuP0nDc0AT5SYON63bus2d0udab0Iw"
TELEGRAM_CHAT_ID   = "314746106"

EXCHANGE_ID   = "okx"
QUOTE_ASSET   = "USDT"
TOP_N_SYMBOLS = 200

RSI_PERIOD = 14
EMA_FAST   = 20
EMA_SLOW   = 50

TIMEFRAMES = {"1h": "Saatlik", "1d": "Günlük"}
TZ_TR = timezone(timedelta(hours=3))
# ============================================================

# Daha önce gönderilen sinyalleri takip et (tekrar gönderme)
sent_signals = set()

def now_tr():
    return datetime.now(TZ_TR).strftime("%d.%m.%Y %H:%M")

def send_telegram(msg):
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    try:
        r = requests.post(url, json={
            "chat_id":    TELEGRAM_CHAT_ID,
            "text":       msg,
            "parse_mode": "Markdown"
        }, timeout=10)
        print(f"  📨 TG: {r.status_code}")
    except Exception as e:
        print(f"  TG Hata: {e}")

def calc_rsi(closes, period=14):
    delta = closes.diff()
    gain  = delta.where(delta > 0, 0.0).ewm(com=period-1, min_periods=period).mean()
    loss  = (-delta.where(delta < 0, 0.0)).ewm(com=period-1, min_periods=period).mean()
    rs    = gain / loss.replace(0, 1e-10)
    return round(float((100 - 100 / (1 + rs)).iloc[-1]), 2)

def calc_ema(closes, period):
    return closes.ewm(span=period, adjust=False).mean()

def get_ohlcv(exchange, symbol, tf, limit=250):
    try:
        raw = exchange.fetch_ohlcv(symbol, timeframe=tf, limit=limit)
        if len(raw) < 60:
            return None
        df = pd.DataFrame(raw, columns=["ts","open","high","low","close","volume"])
        df[["close","volume"]] = df[["close","volume"]].astype(float)
        return df
    except:
        return None

def get_signal_strength(rsi, vol_x, price, ema20, ema50):
    """Sinyalin ne kadar erken/güçlü olduğunu hesapla"""
    score = 0
    # RSI bölgesi
    if 40 <= rsi <= 55:
        score += 3   # En erken bölge — henüz ivme başlıyor
    elif 55 < rsi <= 65:
        score += 2   # Orta bölge
    elif 65 < rsi <= 75:
        score += 1   # Geç bölge ama devam edebilir

    # Hacim katkısı
    if vol_x >= 3.0:
        score += 3
    elif vol_x >= 2.0:
        score += 2
    elif vol_x >= 1.5:
        score += 1

    # EMA yakınlığı — fiyat EMA20'ye ne kadar yakın?
    dist_pct = ((price - ema20) / ema20) * 100
    if dist_pct <= 1.0:
        score += 2   # Henüz yeni kırdı, erken giriş
    elif dist_pct <= 3.0:
        score += 1

    # EMA20-EMA50 yakınlığı — yeni crossover mı?
    ema_gap = ((ema20 - ema50) / ema50) * 100
    if ema_gap <= 1.0:
        score += 2   # EMA'lar yeni ayrışmaya başladı = erken
    elif ema_gap <= 3.0:
        score += 1

    return score

def score_to_stars(score):
    if score >= 8:
        return "⭐⭐⭐ GÜÇLÜ"
    elif score >= 5:
        return "⭐⭐ ORTA"
    else:
        return "⭐ ZAYIF"

def scan_all(exchange, symbols, tf, label):
    hits = []
    print(f"\n  [{label}] taranıyor — {len(symbols)} coin...")

    for sym in symbols:
        df = get_ohlcv(exchange, sym, tf)
        if df is None:
            continue

        c = df["close"]
        v = df["volume"]
        ema20 = calc_ema(c, EMA_FAST)
        ema50 = calc_ema(c, EMA_SLOW)

        try:
            rsi = calc_rsi(c)
        except:
            continue

        price    = float(c.iloc[-1])
        open_    = float(df["open"].iloc[-1])   # Mevcut mumun açılışı
        vol_last = float(v.iloc[-1])
        vol_avg  = float(v.iloc[-20:].mean())
        vol_x    = round(vol_last / vol_avg, 2)

        e20 = float(ema20.iloc[-1])
        e50 = float(ema50.iloc[-1])

        # ─── ERKEN SİNYAL KRİTERLERİ ───────────────────────
        # 1) Fiyat EMA20 üzerinde (veya henüz çok az altında — önceden yakala)
        cond1 = price > e20 * 0.998    # %0.2 tolerans = kırılmak üzere

        # 2) RSI — aşırı alım bölgesinden ÖNCE
        cond2 = 40 <= rsi <= 75        # 75'e kadar izin ver (89 çok geç)

        # 3) EMA20 > EMA50 (yukarı trend) VEYA yeni kesiyor
        cond3 = e20 >= e50 * 0.998     # %0.2 tolerans = kesmek üzere

        # 4) Hacim artışı
        cond4 = vol_x >= 1.2           # 1.2x yeterli (daha erken)

        # 5) Mevcut mum yeşil mi? (alıcı baskısı var)
        cond5 = price > open_

        if cond1 and cond2 and cond3 and cond4 and cond5:
            score = get_signal_strength(rsi, vol_x, price, e20, e50)

            # Aynı sinyali tekrar gönderme
            sig_key = f"{sym}_{tf}_{datetime.now(TZ_TR).strftime('%Y%m%d%H')}"
            if sig_key in sent_signals:
                continue
            sent_signals.add(sig_key)

            hits.append({
                "symbol": sym.replace("/USDT", ""),
                "price":  price,
                "rsi":    rsi,
                "vol_x":  vol_x,
                "score":  score,
                "stars":  score_to_stars(score),
                "dist":   round(((price - e20) / e20) * 100, 2),
                "ema_gap": round(((e20 - e50) / e50) * 100, 2),
            })

        time.sleep(0.06)

    print(f"  → {len(hits)} sinyal")

    zaman = now_tr()

    if not hits:
        msg = (
            f"━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"🤖 *FETHİNHO AI* | OKX Tarama\n"
            f"━━━━━━━━━━━━━━━━━━━━━━━\n"
            f"📊 *{label} Tarama* `[{tf}]`\n"
            f"🕐 {zaman} (UTC+3)\n\n"
            f"❌ Sinyal yok."
        )
    else:
        # Güçlü sinyaller önce
        hits_sorted = sorted(hits, key=lambda x: x["score"], reverse=True)
        lines = [
            "━━━━━━━━━━━━━━━━━━━━━━━",
            "🤖 *FETHİNHO AI* | OKX Tarama",
            "━━━━━━━━━━━━━━━━━━━━━━━",
            f"📊 *{label} Tarama* `[{tf}]`",
            f"🕐 {zaman} (UTC+3)",
            f"✅ *{len(hits)} Sinyal*",
            "───────────────────────",
        ]
        for h in hits_sorted:
            lines.append(
                f"{h['stars']}\n"
                f"🔸 *{h['symbol']}*\n"
                f"   💰 {h['price']:.4f} USDT\n"
                f"   📈 RSI: {h['rsi']}  |  🔥 Hacim: {h['vol_x']}x\n"
                f"   📐 EMA20 üzeri: +{h['dist']}%  |  EMA gap: {h['ema_gap']}%\n"
            )
        lines += [
            "───────────────────────",
            "📌 Erken sinyal sistemi aktif",
            "━━━━━━━━━━━━━━━━━━━━━━━",
        ]
        msg = "\n".join(lines)

    print(msg)
    send_telegram(msg)
    return hits

# ─── BAŞLAT ───────────────────────────────────────────────
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"🤖 FETHİNHO AI — Erken Sinyal Sistemi")
print(f"🕐 {now_tr()} (UTC+3)")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━")

exchange = getattr(ccxt, EXCHANGE_ID)({"enableRateLimit": True})
exchange.load_markets()

all_syms = [
    s for s, m in exchange.markets.items()
    if m.get("quote") == QUOTE_ASSET
    and m.get("active")
    and m.get("type", "spot") == "spot"
    and "/" in s and ":" not in s
]
print(f"   Toplam spot: {len(all_syms)}")

try:
    tickers = exchange.fetch_tickers(all_syms[:400])
    syms = sorted(
        [s for s in tickers if tickers[s].get("quoteVolume")],
        key=lambda s: tickers[s]["quoteVolume"],
        reverse=True
    )[:TOP_N_SYMBOLS]
except:
    syms = all_syms[:TOP_N_SYMBOLS]

print(f"   {len(syms)} coin taranacak\n")

for tf, label in TIMEFRAMES.items():
    scan_all(exchange, syms, tf, label)
    time.sleep(3)

print(f"\n✅ Tamamlandı — {now_tr()}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 FETHİNHO AI — Erken Sinyal Sistemi
🕐 23.04.2026 11:53 (UTC+3)
━━━━━━━━━━━━━━━━━━━━━━━━━━━
   Toplam spot: 292
   200 coin taranacak


  [Saatlik] taranıyor — 200 coin...
  → 11 sinyal
━━━━━━━━━━━━━━━━━━━━━━━
🤖 *FETHİNHO AI* | OKX Tarama
━━━━━━━━━━━━━━━━━━━━━━━
📊 *Saatlik Tarama* `[1h]`
🕐 23.04.2026 11:54 (UTC+3)
✅ *11 Sinyal*
───────────────────────
⭐⭐⭐ GÜÇLÜ
🔸 *CC*
   💰 0.1519 USDT
   📈 RSI: 48.45  |  🔥 Hacim: 1.55x
   📐 EMA20 üzeri: +-0.12%  |  EMA gap: -0.06%

⭐⭐⭐ GÜÇLÜ
🔸 *SENT*
   💰 0.0177 USDT
   📈 RSI: 53.95  |  🔥 Hacim: 2.27x
   📐 EMA20 üzeri: +1.09%  |  EMA gap: -0.02%

⭐⭐ ORTA
🔸 *MASK*
   💰 0.5017 USDT
   📈 RSI: 69.96  |  🔥 Hacim: 6.33x
   📐 EMA20 üzeri: +2.96%  |  EMA gap: 0.95%

⭐⭐ ORTA
🔸 *ZRX*
   💰 0.1128 USDT
   📈 RSI: 58.02  |  🔥 Hacim: 2.9x
   📐 EMA20 üzeri: +1.27%  |  EMA gap: 0.14%

⭐⭐ ORTA
🔸 *GMX*
   💰 7.0390 USDT
   📈 RSI: 65.6  |  🔥 Hacim: 3.73x
   📐 EMA20 üzeri: +1.9%  |  EMA gap: 2.01%

⭐⭐ ORTA
🔸 *NMR*
   💰 9.5530 USDT
   📈 RSI: 72.04